In [49]:
import pandas as pd
import os

In [50]:
ratings = pd.read_csv('../data/preprocessed/ratings_cleaned.csv')
movies = pd.read_csv('../data/preprocessed/movies_cleaned.csv')

In [51]:
ratings.head()

,userId,movieId,rating,timestamp
0,36,145,3.5,2011-06-07 01:33:55
1,36,163,3.0,2011-06-07 01:30:22
2,36,196,3.0,2011-06-07 01:32:53
3,36,485,1.5,2011-06-07 01:32:56
4,36,555,2.0,2011-06-07 01:30:13


In [52]:
user_item_matrix = ratings.pivot(
    index='movieId',
    columns='userId',
    values='rating'
).fillna(0)

In [53]:
user_item_matrix

userId,36,90,118,120,129,140,162,196,199,202,...,138315,138320,138333,138345,138393,138399,138402,138409,138449,138489
movieId,,,,,,,,,,,,,,,,,,,,,
1,0.0,3.5,0.0,0.0,4.0,4.0,3.0,0.0,4.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,4.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118930,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
119141,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
119145,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [54]:
user_item_matrix.shape

(9222, 6914)

In [55]:
output_dir = '../data/preprocessed'

In [56]:
os.makedirs(output_dir, exist_ok=True)

In [57]:
output_path = f'{output_dir}/user_item_matrix_baseline.pkl'
user_item_matrix.to_pickle(output_path)

In [58]:
output_path

'../data/preprocessed/user_item_matrix_baseline.pkl'

## LightGBM preprocessing

In [59]:
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
import ast

In [60]:
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)').astype(float)
movies['year'] = movies['year'].fillna(movies['year'].median())

In [61]:
movies['genres'] = movies['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

In [62]:
mlb = MultiLabelBinarizer()

In [63]:
genre_matrix = mlb.fit_transform(movies['genres'])

In [64]:
genre_df = pd.DataFrame(genre_matrix, columns=[f"gen_{c}" for c in mlb.classes_])

In [65]:
movies_features = pd.concat([movies[['movieId', 'year']], genre_df], axis=1)

In [66]:
movies_features.sample(5)

,movieId,year,gen_Action,gen_Adventure,gen_Animation,gen_Children,gen_Comedy,gen_Crime,gen_Documentary,gen_Drama,...,gen_Film-Noir,gen_Horror,gen_IMAX,gen_Musical,gen_Mystery,gen_Romance,gen_Sci-Fi,gen_Thriller,gen_War,gen_Western
3122,3751,2000.0,0,0,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1202,1428,1995.0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
641,731,1996.0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
8249,73321,2010.0,1,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
6212,8985,2004.0,1,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,1,0,0


In [67]:
def generate_negative_samples(ratings, movies_df, ratio=4):
    """
    For each real interaction (watched movie), generate random negative samples 
    (movies the user has NOT watched) to teach the model what the user ignores.
    """
    user_ids = ratings['userId'].unique()
    movie_ids = movies_df['movieId'].unique()
    
    negatives = []
    
    for u_id in user_ids:
        # Movies the user has already watched (positive interactions)
        seen_movies = set(ratings[ratings['userId'] == u_id]['movieId'])
        
        # Movies the user has NEVER watched
        unseen_movies = list(set(movie_ids) - seen_movies)
        
        # Randomly select negative samples (e.g., 4 times the number of positive samples)
        num_neg = min(len(seen_movies) * ratio, len(unseen_movies))
        neg_samples = np.random.choice(unseen_movies, num_neg, replace=False)
        
        for m_id in neg_samples:
            negatives.append([u_id, m_id, 0]) # 0 represents the "not watched" target
            
    return pd.DataFrame(negatives, columns=['userId', 'movieId', 'target'])

In [68]:
# 1. Create positive samples from actual user ratings
ratings['user_mean'] = ratings.groupby('userId')['rating'].transform('mean')

# We consider that a film is positiv only when rating is higher than user avg 
good_ratings = ratings[ratings['rating'] >= ratings['user_mean']]

positive_samples = good_ratings[['userId', 'movieId']].copy()
positive_samples['target'] = 1

In [69]:
# 2. Generate negative samples
negative_samples = generate_negative_samples(ratings, movies)

In [70]:
# 3. Combine positive and negative samples into one dataset, then shuffle it
full_dataset = pd.concat([positive_samples, negative_samples], axis=0).sample(frac=1).reset_index(drop=True)

In [71]:
# 4. Merge the interaction dataset with the movie features (year and genres)
final_train_df = full_dataset.merge(movies_features, on='movieId', how='left')

In [72]:
# 5. Save the final datasets for the LightGBM model training phase
final_train_df.to_pickle('../data/preprocessed/train_data_lgbm.pkl')
movies_features.to_pickle('../data/preprocessed/movies_features_lgbm.pkl')

In [73]:
final_train_df.sample(5)

,userId,movieId,target,year,gen_Action,gen_Adventure,gen_Animation,gen_Children,gen_Comedy,gen_Crime,...,gen_Film-Noir,gen_Horror,gen_IMAX,gen_Musical,gen_Mystery,gen_Romance,gen_Sci-Fi,gen_Thriller,gen_War,gen_Western
2149402,37772,6979,1,1983.0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,0
3019036,112498,96655,0,2012.0,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0
2056145,130826,1006,0,1996.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2894357,39085,2094,0,1991.0,1,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
246393,31139,8332,0,1971.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [74]:
final_train_df.shape

(4440630, 23)